<a href="https://colab.research.google.com/github/velchan15/MachineLearning-InternshipStarter-FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Markdown — Signal Check 1 (staleness → tied to FlyRank's refresh flag):**

Signal: freshness_tier (days since last update) vs. decline rate.

| freshness_tier  | decline rate | n     |
|-----------------|--------------|-------|
| 91-180          | 0.611        | 9,171 |
| 31-90           | 0.589        | 175   |
| 0-30            | 0.511        | 20,480|
| 181+            | 0.471        | 174   |

Base rate (overall decline rate): 0.542

Verdict: MIXED. The naive assumption behind refresh flags is "older = more likely declining" — but the oldest bucket (181+) actually has the LOWEST decline rate (0.471, below base rate), while the middle bucket (91-180) has the highest (0.611). Staleness alone is not a clean, monotonic signal here. This is a useful negative finding — it means the rule can't rely on freshness alone and needs to be combined
with a visibility check.

**Markdown — Signal Check 2 (volume → tied to FlyRank's quick-win flag):**

Signal: impression_tier (traffic volume) vs. decline rate.

| impression_tier  | decline rate | n      |
|------------------|--------------|--------|
| moderate         | 0.615        | 10,469 |
| good             | 0.586        | 7,205  |
| excellent        | 0.462        | 1,078  |
| low              | 0.454        | 11,248 |

Verdict: MIXED. No clean linear relationship — both the lowest and highest volume tiers have below-base-rate decline, while the middle tiers (moderate, good) are above base rate. This rules out "more traffic = more likely declining" as a simple rule, but it does justify restricting the rule to pages with real visibility (moderate/good/excellent), since a declining page nobody sees isn't worth reviewing.

**Markdown — The rule, in plain words:**

A page is worth reviewing if it's gotten stale (91+ days since last update) AND it still has real visibility (moderate impressions or better) — staleness alone isn't reliable (see Signal Check 1), but combined with a visibility floor it becomes a defensible starting point: don't waste review slots on pages nobody sees anyway.

In [14]:
import os
import urllib.request

csv_path = 'data/raw/content_refresh_anonymized.csv'

# Delete the old dummy file if it exists, then force a fresh download
if os.path.exists(csv_path):
    os.remove(csv_path)

os.makedirs(os.path.dirname(csv_path), exist_ok=True)
url = "https://raw.githubusercontent.com/velchan15/MachineLearning-InternshipStarter-FlyRank/main/data/raw/content_refresh_anonymized.csv"
urllib.request.urlretrieve(url, csv_path)

import pandas as pd
df = pd.read_csv(csv_path)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")

Loaded 30,000 rows, 44 columns


In [15]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

stale = df['freshness_tier'].isin(['91-180', '181+']).astype(int)
visible = df['impression_tier'].isin(['moderate', 'good', 'excellent']).astype(int)

df['score'] = stale * visible * df['impressions_prev_30d']
df['reason_code'] = 'stale_but_visible'
df['action'] = 'review_for_refresh'

print(f"Rows with a non-zero score: {(df['score'] > 0).sum():,} / {len(df):,}")

Rows with a non-zero score: 7,230 / 30,000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
import os

ranked = df.sort_values('score', ascending=False).reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
ranked[['content_id', 'client_id', 'score', 'reason_code', 'action']].to_csv(
    'work/outputs/baseline_action_score.csv', index=False
)

print(f"Wrote {len(ranked):,} ranked rows")
ranked.head(10)[['content_id', 'score', 'reason_code', 'action']]

Wrote 30,000 ranked rows


,content_id,score,reason_code,action
0,content_5fe46e04994d,218786,stale_but_visible,review_for_refresh
1,content_9532f197bbc8,174235,stale_but_visible,review_for_refresh
2,content_2c2606c5d176,164079,stale_but_visible,review_for_refresh
3,content_2dba2b1f9536,137909,stale_but_visible,review_for_refresh
4,content_cb112fce36be,124500,stale_but_visible,review_for_refresh
5,content_c8e9d6ab9013,111885,stale_but_visible,review_for_refresh
6,content_b28d1efd668f,110679,stale_but_visible,review_for_refresh
7,content_36ff89c8214e,106412,stale_but_visible,review_for_refresh
8,content_813e88069237,94762,stale_but_visible,review_for_refresh
9,content_b511d4bc4ad2,83271,stale_but_visible,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [22]:
ranked_eval = df.sort_values('score', ascending=False).reset_index(drop=True)

print("precision@10:", precision_at_k(ranked_eval['score'], ranked_eval['is_declining'], 10))
print("precision@50:", precision_at_k(ranked_eval['score'], ranked_eval['is_declining'], 50))
print("base rate:", df['is_declining'].mean())

ranked_eval.head(20)[['content_id', 'score', 'avg_position', 'ctr', 'is_declining']]

precision@10: 0.6
precision@50: 0.46
base rate: 0.5420666666666667


,content_id,score,avg_position,ctr,is_declining
0,content_5fe46e04994d,218786,4.2,0.14,1
1,content_9532f197bbc8,174235,2.0,0.87,1
2,content_2c2606c5d176,164079,4.2,0.53,1
3,content_2dba2b1f9536,137909,27.9,0.21,0
4,content_cb112fce36be,124500,5.6,0.16,1
5,content_c8e9d6ab9013,111885,9.7,0.00,1
6,content_b28d1efd668f,110679,26.2,0.06,0
7,content_36ff89c8214e,106412,7.3,0.05,0
8,content_813e88069237,94762,26.2,0.06,1
9,content_b511d4bc4ad2,83271,27.9,0.14,0


1. content_5fe46e04994d — flagged for review_for_refresh. Stale (91-180d) + excellent visibility + top score. Confirmed declining. Would be wrong if this page's high impressions are seasonal, not evergreen — refreshing a seasonal page off-season wastes the slot.
2. content_9532f197bbc8 — same reason code, second-highest score. Confirmed declining. Would be wrong if the traffic is bot/spam-inflated rather than real visitors.
3. content_2c2606c5d176 — same pattern, confirmed declining. Would be wrong if the page was already scheduled for a redesign, making a content-only refresh pointless.
4. content_2dba2b1f9536 — flagged, but NOT actually declining (false positive). avg_position is 27.9 — already poor. Would be right if we'd asked "is this page worth fixing at all" instead of "is this page declining" — the rule conflates staleness with decline, but this page may just always have been weak.
5. content_cb112fce36be — confirmed declining, good position (5.6). Strong pick.
6. content_c8e9d6ab9013 — confirmed declining despite 0% CTR — a real quick-win candidate (good position, no clicks — points at title/meta problem).
7. content_b28d1efd668f — NOT declining (false positive). Position 26.2 — same
pattern as #4: poor position going in, so "decline" isn't the real story here.
8. content_36ff89c8214e — NOT declining. Position 7.3 but CTR only 0.05 — this page might genuinely need a CTR fix, not a content refresh; reason code is misleading.
9. content_813e88069237 — confirmed declining, weak position (26.2).
10. content_b511d4bc4ad2 — NOT declining. Same poor-position pattern as #4/#7.
11. content_c21024970297 — NOT declining (false positive). Good position (5.1), decent CTR (0.41%) — this looks like a stable performer, not a candidate for refresh. Rule can't distinguish "stale but doing fine" from "stale and failing."
12. content_d17681677e69 — NOT declining. Position 5.8, CTR 0.24% — same pattern as #11, a healthy page wrongly flagged just for being old + visible.
13. content_89fcb6f35525 — confirmed declining, good position (4.7). Strong pick.
14. content_a7427266c305 — NOT declining. Good position (5.7) but low CTR (0.11%) — this is a CTR/snippet problem, not a staleness problem; reason code mismatch.
15. content_3d94572c3a35 — confirmed declining, good position (4.3). Strong pick.
16. content_11fcfd65d94c — confirmed declining, good position (6.2). Strong pick.
17. content_91652435f57a — NOT declining. Position 7.8, CTR only 0.06% — same CTR-fix mismatch as #14 and #8.
18. content_05e9b4cd9ccf — confirmed declining, weak position (22.1). Correct call.
19. content_40fb6f005d61 — confirmed declining, weak position (26.0). Correct call.
20. content_01908772c6db — confirmed declining, strong position (4.0). Strong pick.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks**

8 of the top 20 flagged pages are false positives (not actually declining), and they split into two distinct failure types:

**Type 1 — pages that were likely always weak, not recently declining (#4, #7, #10, #17 from the top-20 list):** these have poor avg_position (22-28), meaning they probably never ranked well to begin with. The rule flags them as "declining" because they're stale and visible, but their real problem isn't decline — it's that they may have always been mediocre. The reason code "stale_but_visible" can't tell "used to be good, now failing" apart from "has always been average."

**Type 2 — pages that are actually healthy or need a different fix (#11, #12, #14):** these have strong avg_position but low CTR. They aren't declining at all — some are stable performers, and the low-CTR ones look more like a title/snippet problem (a CTR-fix case) than a content-freshness problem. The rule has no way to distinguish these from genuine refresh candidates, because it only checks staleness and visibility, not CTR relative to position.

What this means for a v2 rule: freshness + visibility alone is not enough to separate "needs refresh" from "needs a CTR fix" from "actually fine." A stronger rule would add a CTR-vs-position check (Signal Check 2 from Section 1 hinted at this) before assigning the review_for_refresh action.

**Leakage check**

The rule's score is built from freshness_tier, impression_tier, and
impressions_prev_30d only. None of these are derived from trend_direction or trend_pct — those columns are the source of is_declining_label and are never used as inputs to the score. No future-window data was used; every input comes from data already observed in the trailing 90-day window. is_declining_label, trend_direction, and trend_pct are used only afterward, to evaluate the rule's precision — never as features feeding the score itself.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.